# Лабораторная работа 1. Биграммная языковая модель

Цель работы — реализовать языковую модель из счётчиков: оценить $P(w_t\mid w_{t-1})$, сгенерировать текст, сравнить правдоподобие и перплексию. Для основной части используются только стандартная библиотека Python. Готовые NLP-модели и библиотеки не применяются.

**Корпус:** четыре тома романа Л. Н. Толстого «Война и мир» из открытой коллекции [RusLit](https://github.com/d0rj/RusLit). Корпус загружается в память при запуске и в репозиторий не записывается.

In [2]:
from math import exp, isinf, log
from random import Random
from urllib.parse import quote
from urllib.request import Request, urlopen
import re

class SimpleCounter:

    def __init__(self, iterable=None):
        self._data = {}
        if iterable is not None:
            self.update(iterable)

    def update(self, iterable):
        for item in iterable:
            self.add(item)

    def add(self, item, amount=1):
        self._data[item] = self._data.get(item, 0) + amount

    def __getitem__(self, item):
        return self._data.get(item, 0)

    def __setitem__(self, item, value):
        self._data[item] = value

    def get(self, item, default=0):
        return self._data.get(item, default)

    def items(self):
        return self._data.items()

    def keys(self):
        return self._data.keys()

    def values(self):
        return self._data.values()

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

    def __bool__(self):
        return bool(self._data)

    def most_common(self, n=None):
        ordered = sorted(self._data.items(), key=lambda pair: pair[1], reverse=True)
        return ordered if n is None else ordered[:n]

SEED = 42
rng = Random(SEED)

## 1. Корпус и токенизация

Токен — последовательность русских букв по регулярному выражению `r"[а-яё]+"`. Текст переводится в нижний регистр. Знаки препинания, цифры и фрагменты на других языках отбрасываются: так модель предсказывает только слова, а не смешивает слова со служебными символами. Последние 10 % последовательности токенов откладываются до обучения и используются только для проверки.

In [3]:
BASE_URL = 'https://raw.githubusercontent.com/d0rj/RusLit/main/prose/Tolstoy/'
FILENAMES = [
    'Война и мир. Том 1.txt',
    'Война и мир. Том 2.txt',
    'Война и мир. Том 3.txt',
    'Война и мир. Том 4.txt',
]

def download_utf8(url):
    request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(request, timeout=60) as response:
        return response.read().decode('utf-8')

urls = [BASE_URL + quote(name) for name in FILENAMES]
corpus_text = '\n'.join(download_utf8(url) for url in urls)
print(f'Загружено символов: {len(corpus_text):,}')
print('Источник:', 'https://github.com/d0rj/RusLit')

Загружено символов: 3,501,795
Источник: https://github.com/d0rj/RusLit


In [4]:
TOKEN_RE = re.compile(r'[а-яё]+')

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

tokens = tokenize(corpus_text)
assert len(tokens) >= 100_000, 'Корпус должен содержать не менее 100 000 слов'

split_at = int(len(tokens) * 0.90)
train_tokens_raw = tokens[:split_at]
test_tokens_raw = tokens[split_at:]

word_counts = SimpleCounter(train_tokens_raw)
print(f'Всего слов в корпусе: {len(tokens):,}')
print(f'Обучающая часть: {len(train_tokens_raw):,} слов (90 %)')
print(f'Отложенная часть: {len(test_tokens_raw):,} слов (10 %)')
print(f'Размер словаря обучающей части: {len(word_counts):,}')
print('\n20 самых частых слов:')
for word, count in word_counts.most_common(20):
    print(f'{word:>15}  {count:>7,}')

Всего слов в корпусе: 502,295
Обучающая часть: 452,065 слов (90 %)
Отложенная часть: 50,230 слов (10 %)
Размер словаря обучающей части: 49,930

20 самых частых слов:
              и   21,417
              в   12,307
             не    8,434
            что    7,866
             он    7,381
             на    6,810
              с    6,334
            как    3,983
            его    3,969
              к    3,567
             то    3,514
              я    3,008
             но    2,663
            она    2,631
            это    2,449
           было    2,441
            все    2,120
             по    2,044
         сказал    1,965
             за    1,946


## 2. Счётчики и вероятности

Для каждого контекста $v$ считаются пары $(v,w)$. Без сглаживания вероятность равна $c(v,w)/c(v)$. При `alpha > 0` применяется сглаживание Лапласа: $(c(v,w)+\alpha)/(c(v)+\alpha V)$. Токен `<unk>` включён в словарь: неизвестные слова из отложенной части отображаются в него, поэтому сглаженная модель остаётся определённой и для слов, отсутствовавших при обучении.

In [5]:
class BigramLanguageModel:
    """Биграммная языковая модель, реализованная только на счётчиках."""

    UNK = '<unk>'

    def __init__(self, seed=42):
        self.rng = Random(seed)
        self.bigram = {}
        self.context_totals = SimpleCounter()
        self.vocabulary = set()
        self.vocab_size = 0

    def fit(self, tokens):
        if len(tokens) < 2:
            raise ValueError('Для обучения нужны хотя бы два токена.')
        self.vocabulary = set(tokens) | {self.UNK}
        self.vocab_size = len(self.vocabulary)
        self.bigram = {}
        self.context_totals = SimpleCounter(tokens[:-1])
        for previous, current in zip(tokens, tokens[1:]):
            if previous not in self.bigram:
                self.bigram[previous] = SimpleCounter()
            self.bigram[previous].add(current)
        return self

    def _known(self, word):
        return word if word in self.vocabulary else self.UNK

    def prob(self, v, w, alpha=0.0):
        """Возвращает P(w | v); alpha=0 означает модель без сглаживания."""
        if alpha < 0:
            raise ValueError('alpha не может быть отрицательным.')
        v, w = self._known(v), self._known(w)
        count_vw = self.bigram.get(v, SimpleCounter())[w]
        count_v = self.context_totals[v]
        if alpha == 0:
            return count_vw / count_v if count_v else 0.0
        return (count_vw + alpha) / (count_v + alpha * self.vocab_size)

    def _distribution(self, v, temperature=1.0):
        """Распределение следующего слова без сглаживания, с температурой."""
        if temperature <= 0:
            raise ValueError('Температура должна быть положительной.')
        v = self._known(v)
        successors = self.bigram.get(v, SimpleCounter())
        total = self.context_totals[v]
        if not successors or total == 0:
            return {}
        scaled = {word: (count / total) ** (1 / temperature)
                  for word, count in successors.items()}
        normalizer = sum(scaled.values())
        return {word: value / normalizer for word, value in scaled.items()}

    def generate(self, start, n_tokens=30, temperature=1.0):
        """Генерирует ровно n_tokens слов, начиная со start."""
        if n_tokens < 1:
            raise ValueError('n_tokens должно быть не меньше 1.')
        result = [self._known(start)]
        while len(result) < n_tokens:
            distribution = self._distribution(result[-1], temperature)
            # Для редкого финального слова без продолжений начинаем новую цепочку.
            if not distribution:
                result[-1] = self.rng.choice(list(self.context_totals))
                distribution = self._distribution(result[-1], temperature)
            words, probabilities = zip(*distribution.items())
            result.append(self.rng.choices(words, weights=probabilities, k=1)[0])
        return result

    def logprob(self, text, alpha=0.0):
        """Сумма log P(w_t | w_{t-1}) для строки или последовательности токенов."""
        sequence = tokenize(text) if isinstance(text, str) else list(text)
        if len(sequence) < 2:
            raise ValueError('Нужно не менее двух токенов.')
        sequence = [self._known(word) for word in sequence]
        total_logprob = 0.0
        for v, w in zip(sequence, sequence[1:]):
            probability = self.prob(v, w, alpha=alpha)
            if probability == 0:
                return float('-inf')
            total_logprob += log(probability)
        return total_logprob

    def perplexity(self, tokens, alpha):
        if len(tokens) < 2:
            raise ValueError('Нужно не менее двух токенов.')
        value = self.logprob(tokens, alpha=alpha)
        return float('inf') if isinf(value) else exp(-value / (len(tokens) - 1))

model = BigramLanguageModel(seed=SEED).fit(train_tokens_raw)
print(f'Контекстов: {len(model.context_totals):,}')
print(f'Разных наблюдённых биграмм: {sum(map(len, model.bigram.values())):,}')
print(f'P(\"и\" | \"что\") = {model.prob("что", "и"):.6f}')

Контекстов: 49,930
Разных наблюдённых биграмм: 268,519
P("и" | "что") = 0.005721


## 3. Генерация

Следующее слово выбирается случайно из распределения строки предыдущего слова. Ниже пять последовательностей по 30 слов от разных стартовых слов. Затем для одного и того же старта сравниваются температуры $T=0.5$, $1$ и $2$: вероятности возводятся в степень $1/T$ и перенормируются.

In [6]:
starts = [word for word, _ in model.context_totals.most_common(5)]
for start in starts:
    generated = model.generate(start, n_tokens=30, temperature=1.0)
    print(f'[{start}]', ' '.join(generated))

[и] и матери взглянула на что вот оно делает тебе господи боже мой сказал он видел почти новое пребывание в эрфурт для которого еще больше чем скрипки заиграли гром победы австрия
[в] в конце доклада за другими интересами и голоса говорившие одни били отвесно сверху глядя на его пьер и я буду иметь какое то время этого сражения все откладываемый вопрос одною
[не] не мог перевести через три года с трепетом кабинет к ней думал ростов а й пехотной дивизией под влиянием не то же отвечала ее голос гость может быть он узнавал
[что] что что то что то связь этого огромного ридикюля это подготовлено бесчисленными так или слишком увлечены и чем дело решенное думал он забавляется тихо отвечала она так как лошади батальонного
[он] он бы поименован был человек который по дорожке заставили его на котором муж невольно поддавался веселому настроению духа войска и всё еще удостоивает быть я ей одной стороны поля сражения


In [7]:
temperature_start = starts[0]
for temperature in (0.5, 1.0, 2.0):
    generated = model.generate(temperature_start, n_tokens=30, temperature=temperature)
    print(f'T = {temperature}:', ' '.join(generated))

T = 0.5: и не мог удержать его и изд г и что то что он был в ней и в этом усатом старом давая чувствовать что то что в комнату и как
T = 1.0: и изд г во всем семейством болконских политический термометр на нее что нибудь необыкновенным сходством с правого или общей улыбке тихона и измученные колеблющиеся люди долго радостно насмешливой радостной и
T = 2.0: и долго уже вмещать в это желательно бы трудясь пошел что велено раздеться и московских магазинов сказал вы работники и соли из петербурга князь сейчас выгнать всех приемах выказывал князь


При $T=0.5$ распределение становится более острым: чаще выбираются самые вероятные продолжения, поэтому текст обычно более шаблонный. При $T=2$ распределение выравнивается, редкие варианты появляются чаще, разнообразие растёт, но локальная связность обычно ухудшается. $T=1$ — исходное распределение счётчиков.

## 4. Правдоподобие и сглаживание

Выбирается предложение из отложенной части, а затем его слова случайно перемешиваются. Без сглаживания достаточно одной отсутствующей биграммы, чтобы вероятность произведения стала нулевой, а $\log P$ — равным $-\infty$. Поэтому сравнить два текста по конечным числам без сглаживания невозможно: обе величины часто равны $-\infty$, хотя число и характер невиданных пар могут различаться.

In [8]:
# Находим предложение, которое целиком начинается в отложенных 10 % текста.
sentence_tokens = [tokenize(part) for part in re.split(r'[.!?]+', corpus_text)]
position = 0
connected_raw = None
for sentence in sentence_tokens:
    if position >= split_at and 10 <= len(sentence) <= 30:
        connected_raw = sentence
        break
    position += len(sentence)

# Это защитный вариант на случай необычной пунктуации источника.
if connected_raw is None:
    connected_raw = test_tokens_raw[:20]

connected = [model._known(word) for word in connected_raw]
shuffled = connected.copy()
rng.shuffle(shuffled)

print('Связный фрагмент:')
print(' '.join(connected))
print('\nТе же слова вперемешку:')
print(' '.join(shuffled))

for alpha in (0.0, 0.1):
    lp_connected = model.logprob(connected, alpha=alpha)
    lp_shuffled = model.logprob(shuffled, alpha=alpha)
    print(f'\nalpha={alpha}: log P(связный) = {lp_connected:.3f}; '
          f'log P(перемешанный) = {lp_shuffled:.3f}')

Связный фрагмент:
ты не можешь себе представить как я бываю несчастна <unk> когда ты такой

Те же слова вперемешку:
бываю я можешь <unk> такой как ты несчастна себе представить ты не когда

alpha=0.0: log P(связный) = -inf; log P(перемешанный) = -inf

alpha=0.1: log P(связный) = -94.471; log P(перемешанный) = -111.433


## 5. Оценка на отложенной части

Перплексия считается только на последних 10 % токенов, которые модель не видела при `fit`: $\mathrm{PPL}=\exp\left(-\frac{1}{N}\sum_t\log P(w_t\mid w_{t-1})\right)$. Чем она меньше, тем лучше модель предсказывает этот тестовый текст.

In [9]:
test_tokens = [model._known(word) for word in test_tokens_raw]
results = []
for alpha in (1.0, 0.1, 0.01):
    log_probability = model.logprob(test_tokens, alpha=alpha)
    ppl = model.perplexity(test_tokens, alpha=alpha)
    results.append((alpha, log_probability, ppl))

print(f'{"alpha":>8} | {"log P(test)":>18} | {"perplexity":>12}')
print('-' * 46)
for alpha, log_probability, ppl in results:
    print(f'{alpha:8.2g} | {log_probability:18.2f} | {ppl:12.3f}')

best_alpha, _, best_ppl = min(results, key=lambda row: row[2])
print(f'\nМинимальная перплексия: {best_ppl:.3f} при alpha = {best_alpha}.')

   alpha |        log P(test) |   perplexity
----------------------------------------------
       1 |         -498604.02 |    20467.971
     0.1 |         -466368.73 |    10773.558
    0.01 |         -448033.34 |     7478.698

Минимальная перплексия: 7478.698 при alpha = 0.01.


Слишком большое сглаживание забирает вероятность у часто встречающихся, хорошо подтверждённых данных вариантов и раздаёт её редким или вообще не наблюдавшимся вариантам. В итоге модель меньше доверяет данным.

# Бонус

## Триграммная модель с откатом на биграммы

Триграмма использует два предыдущих слова как контекст. Когда конкретная тройка $(u,v,w)$ не встретилась в обучающем тексте, используется биграммная вероятность $P(w\mid v)$.

In [10]:
class TrigramBackoffLanguageModel(BigramLanguageModel):
    """Триграммы с откатом на биграммы для невиданных троек."""

    def fit(self, tokens):
        super().fit(tokens)
        self.trigram = {}
        self.trigram_totals = SimpleCounter()
        for u, v, w in zip(tokens, tokens[1:], tokens[2:]):
            context = (u, v)
            if context not in self.trigram:
                self.trigram[context] = SimpleCounter()
            self.trigram[context].add(w)
            self.trigram_totals.add(context)
        return self

    def prob3(self, u, v, w, alpha=0.0):
        u, v, w = self._known(u), self._known(v), self._known(w)
        count = self.trigram.get((u, v), SimpleCounter())[w]
        if count == 0:
            return self.prob(v, w, alpha=alpha)
        total = self.trigram_totals[(u, v)]
        return count / total if alpha == 0 else (count + alpha) / (total + alpha * self.vocab_size)

    def _distribution3(self, u, v, temperature=1.0):
        if temperature <= 0:
            raise ValueError('Температура должна быть положительной.')
        u, v = self._known(u), self._known(v)
        successors = self.trigram.get((u, v), SimpleCounter())
        if not successors:
            return self._distribution(v, temperature)
        total = self.trigram_totals[(u, v)]
        scaled = {word: (count / total) ** (1 / temperature)
                  for word, count in successors.items()}
        normalizer = sum(scaled.values())
        return {word: value / normalizer for word, value in scaled.items()}

    def generate(self, start, n_tokens=30, temperature=1.0):
        start_tokens = tokenize(start) if isinstance(start, str) else list(start)
        if len(start_tokens) < 2:
            raise ValueError('Для триграммного генератора нужны два стартовых слова.')
        result = [self._known(word) for word in start_tokens[:2]]
        while len(result) < (n_tokens + 2):
            distribution = self._distribution3(result[-2], result[-1], temperature)
            if not distribution:
                result[-1] = self.rng.choice(list(self.context_totals))
                distribution = self._distribution3(result[-2], result[-1], temperature)
            words, probabilities = zip(*distribution.items())
            result.append(self.rng.choices(words, weights=probabilities, k=1)[0])
        return result

    def logprob(self, text, alpha=0.0):
        sequence = tokenize(text) if isinstance(text, str) else list(text)
        if len(sequence) < 2:
            raise ValueError('Нужно не менее двух токенов.')
        sequence = [self._known(word) for word in sequence]
        first_probability = self.prob(sequence[0], sequence[1], alpha=alpha)
        if first_probability == 0:
            return float('-inf')
        total_logprob = log(first_probability)
        for u, v, w in zip(sequence, sequence[1:], sequence[2:]):
            probability = self.prob3(u, v, w, alpha=alpha)
            if probability == 0:
                return float('-inf')
            total_logprob += log(probability)
        return total_logprob

    def perplexity(self, tokens, alpha):
        value = self.logprob(tokens, alpha=alpha)
        return float('inf') if isinf(value) else exp(-value / (len(tokens) - 1))


trigram_model = TrigramBackoffLanguageModel(seed=SEED).fit(train_tokens_raw)
trigram_ppl = trigram_model.perplexity(test_tokens, alpha=best_alpha)
bigram_ppl = model.perplexity(test_tokens, alpha=best_alpha)
print(f'alpha для сравнения: {best_alpha}')
print(f'Биграммная перплексия: {bigram_ppl:.3f}')
print(f'Триграммная перплексия с откатом: {trigram_ppl:.3f}')

alpha для сравнения: 0.01
Биграммная перплексия: 7478.698
Триграммная перплексия с откатом: 7799.326


In [11]:
start_pair = max(trigram_model.trigram_totals, key=trigram_model.trigram_totals.get)
print('Стартовая пара:', ' '.join(start_pair))
print('\nБиграмма:')
print(' '.join(model.generate(start_pair[-1], n_tokens=30, temperature=1.0)))
print('\nТриграмма с откатом:')
print(' '.join(trigram_model.generate(start_pair, n_tokens=30, temperature=1.0)))

Стартовая пара: что он

Биграмма:
он треснуло дерево увидав князя о том что всегда бывает постояннее прибавил государь и сказать это сказала княжна марья и совестно было знать потому возможностью поправить это не пригласили ее

Триграмма с откатом:
что он их громко выговорит а что ежели ударить теперь с ним ну деньги выйдут тогда что там по ту сторону моста против неприятеля под командою в это время в особенности тщеславие


При α = 0.01 биграммная модель показала меньшую перплексию: 7478.698 против 7799.326 у триграммной модели с откатом. Вероятно, триграммы оказались слишком разреженными: точные контексты из двух слов повторяются реже, поэтому модели часто приходится возвращаться к биграммам. При этом триграммная генерация лучше сохраняет стартовый контекст «что он» и иногда создаёт более локально связные фрагменты, но всё ещё не удерживает общий смысл текста.

## Сравнение с Qwen3-0.6B-Base

Следующая ячейка считает логарифмы вероятностей тех же связного и перемешанного фрагментов у `Qwen/Qwen3-0.6B-Base`.

In [12]:
import os, math, torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

plt.rcParams.update({"figure.dpi": 110, "font.size": 12})
os.makedirs("fallback", exist_ok=True)

MODEL_ID = os.environ.get("TGO_MODEL_ID", "Qwen/Qwen3-0.6B-Base")

device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print("device:", device)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model_qwen = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float32
    )
except TypeError:
    model_qwen = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32
    )  # transformers 4.x
model_qwen = model_qwen.to(device).eval()
print(f"модель: {MODEL_ID}, параметров: {sum(p.numel() for p in model_qwen.parameters())/1e6:.0f} млн, словарь: {len(tok)} токенов")

device: cuda


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

модель: Qwen/Qwen3-0.6B-Base, параметров: 596 млн, словарь: 151669 токенов


In [13]:
@torch.no_grad()
def qwen_logprob(words):
    encoded = tok(' '.join(words), return_tensors='pt')
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    logits = model_qwen(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1, :]
    target_ids = input_ids[:, 1:]
    token_logprobs = torch.log_softmax(logits, dim=-1)
    selected = token_logprobs.gather(2, target_ids.unsqueeze(-1)).squeeze(-1)
    return selected.sum().item(), target_ids.numel()


qwen_connected, n_connected = qwen_logprob(connected)
qwen_shuffled, n_shuffled = qwen_logprob(shuffled)
bigram_connected = model.logprob(connected, alpha=best_alpha)
bigram_shuffled = model.logprob(shuffled, alpha=best_alpha)

print('Модель              | связный       | перемешанный  | разрыв')
print('-' * 70)
print(f'Биграмма (alpha={best_alpha}) | {bigram_connected:13.3f} | {bigram_shuffled:13.3f} | {bigram_connected - bigram_shuffled:13.3f}')
print(f'Qwen3-0.6B-Base    | {qwen_connected:13.3f} | {qwen_shuffled:13.3f} | {qwen_connected - qwen_shuffled:13.3f}')
print(f'Число предсказанных токенов Qwen: {n_connected} и {n_shuffled}')

Модель              | связный       | перемешанный  | разрыв
----------------------------------------------------------------------
Биграмма (alpha=0.01) |       -82.391 |      -109.861 |        27.470
Qwen3-0.6B-Base    |       -86.965 |      -117.663 |        30.699
Число предсказанных токенов Qwen: 21 и 21


Обе модели оценили связный фрагмент выше, чем те же слова в случайном порядке. У биграмм разрыв составил 27.470, а у Qwen — 30.699, поэтому Qwen сильнее реагирует на нарушение естественного порядка слов.

## Наивный Байес на частотах слов

Второй корпус — Ф. М. Достоевский, «Братья Карамазовы», из той же открытой коллекции. Класс предложения выбирается по наибольшему значению $\log P(	ext{автор}) + \sum \log P(w\mid	ext{автор})$.


In [14]:
DOSTOEVSKY_URL = ('https://raw.githubusercontent.com/d0rj/RusLit/main/prose/Dostoevsky/'
                  + quote('Братья Карамазовы.txt'))
dostoevsky_text = download_utf8(DOSTOEVSKY_URL)
dostoevsky_tokens = tokenize(dostoevsky_text)
assert len(dostoevsky_tokens) >= 100_000
dost_split = int(len(dostoevsky_tokens) * 0.90)
dost_train = dostoevsky_tokens[:dost_split]

class NaiveBayesAuthor:
    UNK = '<unk>'

    def fit(self, texts_by_author):
        self.authors = list(texts_by_author)
        self.counts = {author: SimpleCounter(tokens) for author, tokens in texts_by_author.items()}
        self.totals = {author: sum(counts.values()) for author, counts in self.counts.items()}
        self.vocabulary = set().union(*(counts.keys() for counts in self.counts.values())) | {self.UNK}
        self.vocab_size = len(self.vocabulary)
        total_tokens = sum(self.totals.values())
        self.log_priors = {author: log(self.totals[author] / total_tokens) for author in self.authors}
        return self

    def score(self, tokens, author, alpha=0.1):
        denominator = self.totals[author] + alpha * self.vocab_size
        score = self.log_priors[author]
        for word in tokens:
            word = word if word in self.vocabulary else self.UNK
            score += log((self.counts[author][word] + alpha) / denominator)
        return score

    def predict(self, tokens, alpha=0.1):
        scores = {author: self.score(tokens, author, alpha) for author in self.authors}
        return max(scores, key=scores.get), scores

author_model = NaiveBayesAuthor().fit({'Толстой': train_tokens_raw, 'Достоевский': dost_train})
print(f'Общий словарь классификатора: {author_model.vocab_size:,}')

Общий словарь классификатора: 67,652


In [16]:
def heldout_sentences(text, split_index, limit=5):
    result, position = [], 0
    for part in re.split(r'[.!?]+', text):
        sentence = tokenize(part)
        if position >= split_index:
            result.append(sentence)
            if len(result) == limit:
                return result
        position += len(sentence)
    return result

classification_set = ([(sentence, 'Толстой') for sentence in heldout_sentences(corpus_text, 10, 10)] +
                      [(sentence, 'Достоевский') for sentence in heldout_sentences(dostoevsky_text, 30, 10)])
assert classification_set, 'Не удалось выделить предложения из отложенных частей.'

correct = 0
for sentence, actual in classification_set:
    predicted, scores = author_model.predict(sentence, alpha=0.1)
    correct += predicted == actual
    print(f'истинный: {actual:12} | предсказан: {predicted:12} | вероятности: {scores} | {" ".join(sentence[:12])}...')

accuracy = correct / len(classification_set)
print(f'\nТочность на {len(classification_set)} отложенных предложениях: {accuracy:.1%}')


истинный: Толстой      | предсказан: Толстой      | вероятности: {'Толстой': -41.02509231943801, 'Достоевский': -54.56581892802626} | война и мир война и мир...
истинный: Толстой      | предсказан: Толстой      | вероятности: {'Толстой': -24.990365000634657, 'Достоевский': -27.510914412577804} | том часть первая...
истинный: Толстой      | предсказан: Толстой      | вероятности: {'Толстой': -12.369524842515396, 'Достоевский': -15.813655643312186} | поместья...
истинный: Толстой      | предсказан: Толстой      | вероятности: {'Толстой': -30.10129861825256, 'Достоевский': -30.394223200840983} | мой верный раб...
истинный: Толстой      | предсказан: Толстой      | вероятности: {'Толстой': -28.275470739830837, 'Достоевский': -28.879447429676922} | ну здравствуйте здравствуйте...
истинный: Толстой      | предсказан: Толстой      | вероятности: {'Толстой': -26.899820849801074, 'Достоевский': -29.583279787715142} | садитесь и рассказывайте...
истинный: Толстой      | предсказан: Толстой      

Классификатор правильно определил автора в 85 % случаев: 17 из 20 отложенных фрагментов. Длинные характерные предложения распознаются хорошо, поскольку содержат больше слов и стилевых признаков, накопленных в частотах автора. Ошибки возникают на очень коротких фрагментах, заголовках, обрывках вроде «ст» или пустой последовательности после токенизации: в них почти нет информации об авторском стиле, поэтому решение определяется близкими оценками или априорной вероятностью.